# מעבדה 00 — פאזורים: שפת הגלים

במעבדה הזו תחברו תנודות על ידי חיבור חִצים — ותשתמשו בחִצים כדי לחזות שקט, פעימות, ואת
ההבדל בין לייזר למנורה.

עבדו לפי הסדר. במקום שבו המחברת מבקשת מכם לנבא, כתבו את הניבוי שלכם בתא המיועד לכך
**לפני** הרצת התא הבא. זה אינו טקס: ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות
שטעיתם.

## מפרט המודל

| | |
|---|---|
| **מערכת** | תנודות הרמוניות בתדירות זוויתית אחת, כל אחת מיוצגת על ידי משרעת מרוכבת (פאזור) |
| **דינמיקה** | סיבוב אחיד בלבד — כל פאזור מסתובב כ-$e^{-i\omega t}$; שום דבר אחר אינו קורה |
| **שפה** | אין; הפאזורים אינם גדלים, אינם דועכים ואינם מחליפים דבר |
| **אנסמבל** | דטרמיניסטי, פרט לניסוי המופעים האקראיים: הגרלות אחידות בלתי תלויות, בעלות זרע |
| **מוזנח** | כל דבר פיזיקלי — מה מתנודד, וכל מנגנון שהיה משנה משרעת בזמן |
| **תקף כאשר** | האותות הן אכן סינוסים בעלי תדירות בודדת |
| **אופני כשל** | סיכום תדירויות שונות כאילו היו פאזורים; פעולות לא-לינאריות המופעלות על האות המרוכב |

כל הפיזיקה נמצאת ב-`wavelab.phasors` — פתחו וקראו אותה; היא קצרה יותר ממחברת זו.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# wavelab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import phasors
from wavelab.constants import SIGN_CONVENTION
from wavelab.validation import scaling_exponent

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

OMEGA = 2 * np.pi  # rad/s — period 1 s, so time axes read directly in cycles
print(SIGN_CONVENTION)

## חלק 1 — חץ מסתובב אחד

התנודה $x(t) = A\cos(\omega t + \varphi)$ היא הצל האופקי של חץ באורך $A$ המסתובב
**בכיוון השעון**: $x(t) = \mathrm{Re}[\hat{x}\, e^{-i\omega t}]$, כאשר הפאזור
$\hat{x} = A e^{-i\varphi}$ קבוע והסיבוב משותף לכל תנודה באותה תדירות.

הריצו את התא הבא כדי לראות חץ אחד ואת צילו.

In [ ]:
amplitude, phase = 1.0, np.pi / 3
z = phasors.phasor(amplitude, phase)
t = np.linspace(0.0, 2.0, 400)
signal = phasors.real_signal(amplitude, OMEGA, phase, t)

fig, (left, right) = plt.subplots(
    1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1, 1.8]}
)
snapshots = phasors.evaluate(z, OMEGA, np.arange(8) / 8.0)
left.axhline(0, color="0.85"), left.axvline(0, color="0.85")
for i, w in enumerate(snapshots):
    left.annotate(
        "", xy=(w.real, w.imag), xytext=(0, 0),
        arrowprops={"arrowstyle": "->", "color": plt.cm.viridis(i / 8)},
    )
left.add_patch(plt.Circle((0, 0), amplitude, fill=False, color="0.9"))
left.set_xlim(-1.2, 1.2), left.set_ylim(-1.2, 1.2), left.set_aspect("equal")
left.set_xlabel("Re"), left.set_ylabel("Im")
left.set_title("one turn in eight snapshots — clockwise")
right.plot(t, signal, lw=1.5)
right.set_xlabel("t (s)"), right.set_ylabel("x(t)")
right.set_title("the shadow: a cosine with the phasor's amplitude and phase")
plt.tight_layout()
plt.show()

## חלק 2 — לחבר שתי תנודות

### לנבא

שתי תנודות בעלות משרעת שווה $a$ ואותה תדירות מחוברות זו לזו.

1. בהפרש מופע $\delta = \pi$: מהי משרעת השקיל?
2. ב-$\delta = \pi/2$?
3. שרטטו את משרעת השקיל כפונקציה של $\delta$ מ-$0$ עד $2\pi$ — היכן נמצאים המקסימום
   שלה והאפסים שלה?

כתבו את הניבוי שלכם **לפני** הרצת התא הבא.

**הניבוי שלכם:**

In [ ]:
a = 1.0
deltas = np.linspace(0.0, 2 * np.pi, 200)
resultants = np.array([phasors.resultant([a, a], [0.0, d])[0] for d in deltas])

# The law of cosines, predicted analytically:
predicted = np.sqrt(2 * a**2 * (1.0 + np.cos(deltas)))

# And the honest brute force: sample each summed signal in time and take its peak.
t_fine = np.linspace(0.0, 1.0, 2001)
brute = np.array(
    [np.max(np.abs(a * np.cos(OMEGA * t_fine) + a * np.cos(OMEGA * t_fine + d))) for d in deltas]
)

plt.figure(figsize=(7, 4))
plt.plot(deltas, resultants, lw=2, label="phasor sum |z1 + z2|")
plt.plot(deltas, predicted, "--", lw=1.2, label="law of cosines")
plt.plot(deltas, brute, ":", lw=1.6, label="brute-force peak of the summed signal")
plt.xlabel("phase difference δ (rad)"), plt.ylabel("resultant amplitude")
plt.legend(), plt.tight_layout()
plt.show()

שלושה חישובים בלתי תלויים, עקומה אחת. סכום שתי משרעות שוות נע בין $2a$ (באותו מופע) דרך
$\sqrt{2}\,a$ (רבע מחזור) ועד בדיוק **אפס** (במופעים מנוגדים): הוספת אות יכולה להשתיק
אות. העקומה הזו *היא* ביטול רעש — ובמודול 08 היא תהיה תבנית ההתאבכות של שני סדקים.

## חלק 3 — פעימות

הַסיטו את התנודה השנייה מעט. עכשיו אין מערכת ייחוס מסתובבת משותפת שאפשר לחלק אותה
החוצה, והסכום אינו קוסינוס כלל: שני החִצים נסחפים בין הסכמה להתנגדות בתדירות ה*הפרש*.

### לנבא

440 הרץ בתוספת 441 הרץ, משרעות שוות: כמה פעימות עוצמה בשנייה? ו-440 הרץ בתוספת 444
הרץ? רשמו את זה, ואז הריצו את התא (מואט פי מאה כדי שתוכלו לראות אותו).

**הניבוי שלכם:**

In [ ]:
f1, f2 = 4.0, 5.0  # Hz — think "440 and 550", slowed a hundredfold to be visible
t_long = np.linspace(0.0, 3.0, 3001)
signal = phasors.beat_signal(1.0, 1.0, 2 * np.pi * f1, 2 * np.pi * f2, t_long)
envelope = 2.0 * np.cos(np.pi * (f1 - f2) * t_long)

plt.figure(figsize=(9, 3.6))
plt.plot(t_long, signal, lw=0.9)
plt.plot(t_long, np.abs(envelope), "--", color="0.5", lw=1.2)
plt.plot(t_long, -np.abs(envelope), "--", color="0.5", lw=1.2)
plt.xlabel("t (s)"), plt.ylabel("summed signal")
plt.title("4 Hz + 5 Hz: a 4.5 Hz carrier, loudness pulsing once per second")
plt.tight_layout()
plt.show()

# Measure the beat period as the spacing of the envelope's zeros, and compare:
crossings = np.nonzero(np.diff(np.signbit(envelope)))[0]
beat_period = float(np.mean(np.diff(t_long[crossings])))
print(f"measured beat period {beat_period:.3f} s   vs   1/|f1-f2| = {1.0 / abs(f1 - f2):.3f} s")

## חלק 4 — מאה כנרים

$N$ פאזורי-יחידה בעלי מופעים אקראיים בלתי תלויים: כמה אורך חץ הסכום, בדרך כלל? הריצו
את התא מחדש — השקיל שונה בכל פעם, אך הגודל ה*טיפוסי* שלו מציית לחוק, והחוק הזה מפריד
מנורה מלייזר.

In [ ]:
sizes = np.array([10, 100, 1000])
n_trials = 200
rms = []
for n in sizes:
    magnitudes = [abs(phasors.random_phasor_sum(int(n), rng)) for _ in range(n_trials)]
    rms.append(float(np.sqrt(np.mean(np.square(magnitudes)))))

exponent = scaling_exponent(sizes, rms)
plt.figure(figsize=(6, 4))
plt.loglog(sizes, rms, "o-", label=f"RMS resultant (fitted slope {exponent:.2f})")
plt.loglog(sizes, np.sqrt(sizes), "--", color="0.5", label="sqrt(N) — random walk")
plt.loglog(sizes, sizes, ":", color="0.7", label="N — phase-locked")
plt.xlabel("number of phasors N"), plt.ylabel("resultant amplitude")
plt.legend(), plt.tight_layout()
plt.show()
print(f"fitted exponent {exponent:.3f}   (random phases: 1/2; locked phases would give 1)")
print(f"intensity, locked vs random, at N=100: {100**2 // 100}x — the case for the laser")

## חלק 5 — בדיקות אוטומטיות

הטענות שהמחברת הזו העלתה, כטענות הרצות (assertions) — אותן בדיקות בדיוק הרצות בחבילת
הבדיקות של הפרויקט, כך שהעמוד, המחברת והספרייה אינם יכולים לסטות זה מזה בשקט.

In [ ]:
# 1. The representation: the phasor round trip reproduces the direct cosine exactly.
t_check = np.linspace(0.0, 5.0, 2001)
via_phasor = phasors.real_signal(0.7, 3.0, 1.1, t_check)
assert np.max(np.abs(via_phasor - 0.7 * np.cos(3.0 * t_check + 1.1))) < 1e-12

# 2. The interference law, at an arbitrary phase difference.
amp, _ = phasors.resultant([0.8, 1.3], [0.0, 0.9])
assert abs(amp - np.sqrt(0.8**2 + 1.3**2 + 2 * 0.8 * 1.3 * np.cos(0.9))) < 1e-12

# 3. Complete cancellation and complete reinforcement.
cancelled, _ = phasors.resultant([1.0, 1.0], [0.0, np.pi])
doubled, _ = phasors.resultant([1.0, 1.0], [0.3, 0.3])
assert cancelled < 1e-12 and abs(doubled - 2.0) < 1e-12

# 4. The beat identity: the honest sum equals carrier times envelope.
w1, w2 = 3.0, 3.4
beat = phasors.beat_signal(0.6, 0.6, w1, w2, t_check)
identity = 2 * 0.6 * np.cos(0.5 * (w1 - w2) * t_check) * np.cos(0.5 * (w1 + w2) * t_check)
assert np.max(np.abs(beat - identity)) < 1e-12

# 5. The random walk: fitted exponent within 0.1 of 1/2, on fresh seeds.
check_rng = np.random.default_rng(1)
check_rms = [
    float(np.sqrt(np.mean([abs(phasors.random_phasor_sum(n, check_rng)) ** 2 for _ in range(100)])))
    for n in (10, 100, 1000)
]
assert abs(scaling_exponent([10, 100, 1000], check_rms) - 0.5) < 0.1

print("all checks passed")

## חלק 6 — לחקור בעצמכם

שני ניסויים שכדאי לעשות:

1. מצאו זוג משרעות ($A_1 \neq A_2$) והפרש מופע שבהם השקיל שווה בדיוק ל-$A_1$ — תנודה
   שנייה קיימת אך בלתי נראית במשרעת.
2. קִבעו $A_2 = 0.99\,A_1$ ו-$\delta = 180°$. עד כמה קטנה השארית? זו המידה שבה ביטול
   רעש חייב להתאים משרעות כדי לזכות בשמו.

In [ ]:
import ipywidgets as widgets


def explore(a1=1.0, a2=0.7, delta_degrees=120.0):
    delta = np.radians(delta_degrees)
    z1, z2 = phasors.phasor(a1, 0.0), phasors.phasor(a2, delta)
    amp, phi = phasors.resultant([a1, a2], [0.0, delta])
    tt = np.linspace(0.0, 2.0, 800)
    reach = max(a1 + a2, 0.1)

    fig, (left, right) = plt.subplots(
        1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1, 1.8]}
    )
    left.axhline(0, color="0.85"), left.axvline(0, color="0.85")
    left.annotate("", xy=(z1.real, z1.imag), xytext=(0, 0),
                  arrowprops={"arrowstyle": "->", "color": "tab:blue"})
    left.annotate("", xy=(z1.real + z2.real, z1.imag + z2.imag), xytext=(z1.real, z1.imag),
                  arrowprops={"arrowstyle": "->", "color": "tab:orange"})
    left.annotate("", xy=(z1.real + z2.real, z1.imag + z2.imag), xytext=(0, 0),
                  arrowprops={"arrowstyle": "->", "color": "tab:green", "lw": 2})
    left.set_xlim(-reach, reach), left.set_ylim(-reach, reach), left.set_aspect("equal")
    left.set_xlabel("Re"), left.set_ylabel("Im")
    left.set_title(f"resultant amplitude {amp:.3f}")
    right.plot(tt, phasors.real_signal(a1, OMEGA, 0.0, tt), lw=0.9, color="tab:blue")
    right.plot(tt, phasors.real_signal(a2, OMEGA, delta, tt), lw=0.9, color="tab:orange")
    right.plot(tt, phasors.real_signal(amp, OMEGA, phi, tt), lw=1.8, color="tab:green")
    right.set_xlabel("t (s)"), right.set_ylim(-reach, reach)
    plt.tight_layout()
    plt.show()


widgets.interact(
    explore,
    a1=widgets.FloatSlider(min=0.0, max=2.0, step=0.05, value=1.0, description="A1"),
    a2=widgets.FloatSlider(min=0.0, max=2.0, step=0.05, value=0.7, description="A2"),
    delta_degrees=widgets.FloatSlider(min=0.0, max=360.0, step=5.0, value=120.0,
                                      description="δ (deg)"),
);

## בדקו את הבנתכם

הריצו את התא שלהלן לחידון עם בדיקה אוטומטית. אותן שאלות, בתוספת הסברים כתובים לכל
אפשרות, נמצאות בעמוד המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "00-phasors.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

כתבו כמה משפטים על כל אחת מהשאלות, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בהיגיון שלכם?
2. המודל הזה מייצג כל תנודה על ידי משרעת מרוכבת *קבועה*. ציינו את הרגע במחברת הזו שבו
   הייצוג הזה קרס, ומה החליף אותו.
3. הסבירו, בלי משוואות, מדוע מאה מקורות בעלי מופע אקראי הם פי מאה ממקור אחד, בעוד מאה
   מקורות נעולי-מופע הם פי עשרת אלפים.

**התשובות שלכם:**

1.
2.
3.